# Deep Learning sur N-Grammes (CNN 1D)

Ce notebook implémente un **CNN 1D** en PyTorch pour classifier les avis Yelp (étoiles 1-5) à partir de features N-grammes.

**Grille** : *Deep-Ngram* → 1 pt | *Arch Deep: plusieurs* → 4 pts

### Architecture CNN 1D
```
Embedding → Conv1d(kernels 3,4,5) → MaxPool → Dense → Output
```

### Checklist SAE-115
- [ ] Charger les features N-grammes
- [ ] Implémenter CNN 1D PyTorch : Embedding → Conv1d (kernels 3,4,5) → MaxPool → Dense → Output
- [ ] Split identique aux autres notebooks
- [ ] Entraînement avec Adam + CrossEntropyLoss
- [ ] Métriques : accuracy, precision, recall, f1, confusion matrix
- [ ] Courbes de loss train/val
- [ ] Sauvegarder le modèle en `.pt`
- [ ] Notebook exécutable sans erreur

## 0. Imports et Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

# Chemins
DATA_DIR   = '../../data/cleaned'
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch version : {torch.__version__}')

## 1. Chargement et Préparation des Données

In [ ]:
print('Chargement des données...')
try:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'), engine='fastparquet')
except Exception:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'))

df = df.dropna(subset=['text', 'stars'])
print(f'Dimensions du dataset : {df.shape}')
print(f"Distribution des étoiles :\n{df['stars'].value_counts(normalize=True).sort_index()}")

In [ ]:
# Échantillonnage — identique à 02-ml-ngram pour comparaison
SAMPLE_SIZE = 5000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42)

X = df_sample['text']
# Labels : 1-5 → 0-4 pour PyTorch
y = df_sample['stars'].astype(int) - 1

# Split 80 / 10 / 10 — identique aux autres notebooks
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

## 2. Vectorisation N-Grammes

Même configuration que `02-ml-ngram.ipynb` : `CountVectorizer` avec `ngram_range=(1,2)` et `max_features=10000`.

In [ ]:
VOCAB_SIZE = 10_000

vectorizer = CountVectorizer(
    max_features=VOCAB_SIZE,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

print(f'Matrice train : {X_train_vec.shape}')

## 3. Dataset PyTorch

On convertit les matrices sparse en tenseurs denses. Pour le CNN 1D, on traite chaque document comme une séquence de longueur 1 avec `VOCAB_SIZE` features.

In [ ]:
class NgramDataset(Dataset):
    """Dataset PyTorch pour les features N-grammes (sparse matrix → dense tensor)."""
    
    def __init__(self, X_sparse, y_labels):
        # Convertir la matrice sparse en array dense float32
        self.X = torch.tensor(X_sparse.toarray(), dtype=torch.float32)
        self.y = torch.tensor(y_labels.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        # Shape: (VOCAB_SIZE,) → unsqueeze → (1, VOCAB_SIZE) pour Conv1d
        return self.X[idx].unsqueeze(0), self.y[idx]


BATCH_SIZE = 64

train_dataset = NgramDataset(X_train_vec, y_train)
val_dataset   = NgramDataset(X_val_vec,   y_val)
test_dataset  = NgramDataset(X_test_vec,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches train : {len(train_loader)} | val : {len(val_loader)} | test : {len(test_loader)}')

## 4. Architecture CNN 1D

Architecture : `Conv1d` parallèle avec kernels de taille **3**, **4** et **5** (à la Kim 2014), suivie d'un `MaxPool`, puis d'un classifieur dense.

```
Input : (batch, 1, VOCAB_SIZE)
  ↓
Conv1d(kernel=3) + ReLU + MaxPool  →  (batch, 128)
Conv1d(kernel=4) + ReLU + MaxPool  →  (batch, 128)
Conv1d(kernel=5) + ReLU + MaxPool  →  (batch, 128)
  ↓  concat
Dense(384 → 128) + ReLU + Dropout
Dense(128 → 5)
```

In [ ]:
class TextCNN1D(nn.Module):
    """CNN 1D pour la classification de texte sur features N-grammes."""
    
    def __init__(self, vocab_size: int, num_classes: int = 5,
                 num_filters: int = 128, kernel_sizes=(3, 4, 5),
                 dropout: float = 0.5):
        super().__init__()
        
        # Convolutions parallèles avec kernels de tailles différentes
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=1,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        
        # Classifieur dense
        fc_in = num_filters * len(kernel_sizes)
        self.fc1 = nn.Linear(fc_in, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x : (batch, 1, vocab_size)
        pooled_outputs = []
        for conv in self.convs:
            h = self.relu(conv(x))          # (batch, num_filters, L)
            h = h.max(dim=2).values         # Global max-pool → (batch, num_filters)
            pooled_outputs.append(h)
        
        out = torch.cat(pooled_outputs, dim=1)  # (batch, num_filters * n_kernels)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return out


model = TextCNN1D(vocab_size=VOCAB_SIZE, num_classes=5).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParamètres entraînables : {total_params:,}')

## 5. Entraînement (Adam + CrossEntropyLoss)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total   += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total   += len(y_batch)
    return total_loss / total, correct / total

In [ ]:
# Hyperparamètres
EPOCHS    = 15
LR        = 1e-3
PATIENCE  = 3   # Early stopping

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    
    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Train loss: {tr_loss:.4f} acc: {tr_acc:.4f} | '
          f'Val loss: {vl_loss:.4f} acc: {vl_acc:.4f}')
    
    # Early stopping
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'cnn1d_ngram_best.pt'))
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping à l\'epoch {epoch}.')
            break

print('\nEntraînement terminé.')

## 6. Courbes de Loss Train / Val

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].set_title('Loss Train / Validation')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history['train_acc'], label='Train', marker='o', color='darkorange')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val',   marker='s', color='steelblue')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Train / Validation')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Évaluation sur le Test Set

On recharge le meilleur modèle (sauvegardé par early stopping).

In [ ]:
# Recharger les meilleurs poids
model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'cnn1d_ngram_best.pt'), map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print('=== RÉSULTATS SUR LE TEST SET ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Macro  : {f1:.4f}')

In [ ]:
# Rapport de classification complet
target_names = [f'{i+1} étoile(s)' for i in range(5)]
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=target_names, yticklabels=target_names)
plt.title('Matrice de Confusion — CNN 1D N-Grammes (Test)')
plt.ylabel('Vrai')
plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Sauvegarde du Modèle Final

In [ ]:
import joblib

# Sauvegarder le modèle final .pt
final_model_path = os.path.join(MODELS_DIR, 'cnn1d_ngram_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': VOCAB_SIZE,
    'num_classes': 5,
    'accuracy_test': acc,
    'f1_test': f1
}, final_model_path)

# Sauvegarder aussi le vectorizer
vec_path = os.path.join(MODELS_DIR, 'count_vectorizer_deep.pkl')
joblib.dump(vectorizer, vec_path)

print(f'Modèle sauvegardé   : {final_model_path}')
print(f'Vectorizer sauvegardé : {vec_path}')
print(f'\nPerformance finale (test) — Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}')

---
## ✅ Résumé

| Étape | Résultat |
|-------|----------|
| Features N-grammes | CountVectorizer(ngram_range=(1,2), max_features=10k) |
| Architecture | CNN 1D — Conv(k=3,4,5) + MaxPool + Dense |
| Optimiseur | Adam (lr=1e-3) + CrossEntropyLoss |
| Early stopping | Patience=3 |
| Modèle sauvegardé | `models/cnn1d_ngram_final.pt` |

**Note** : Si PyTorch n'est pas installé, lancer :
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
```